## Reproducing ResNet on CIFAR-10: Experiments on Network Depth, Batch Size, and Pooling

---
- Baseline Code Link: https://github.com/kuangliu/pytorch-cifar

In [1]:
'''Train CIFAR10 with PyTorch.'''
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

from torchsummary import summary

import os
import argparse

import matplotlib.pyplot as plt
import numpy as np

# from resnet_20_32_44_56_v1 import *   # identity mapping shortcut version 1
from resnet_20_32_44_56_v2 import *   # identity mapping shortcut version 2
from utils import progress_bar

In [2]:
parser = argparse.ArgumentParser(description='PyTorch CIFAR10 Training')
parser.add_argument('--lr', default=0.1, type=float, help='learning rate')
parser.add_argument('--resume', '-r', action='store_true',
                    help='resume from checkpoint')
# args = parser.parse_args()
args, _ = parser.parse_known_args()

device = torch.device("mps") if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")
best_acc = 0   # best test accuracy
start_epoch = 0   # start from epoch 0 or last checkpoint epoch

device: mps


In [3]:
# Data
print('==> Preparing data..')
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=100, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

==> Preparing data..
Files already downloaded and verified
Files already downloaded and verified


In [4]:
# Model
print('==> Building Model..\n')

net = ResNet20()
# net = ResNet32()
# net = ResNet44()
# net = ResNet56()

# Check layer(type), output shape, param #
print('==> Model Summary')
summary(net, (3, 32, 32))

net = net.to(device)

if args.resume:
    # Load checkpoint.
    print('==> Resuming from checkpoint..')
    assert os.path.isdir('checkpoint'), 'Error: no checkpoint directory found!'
    checkpoint = torch.load('./checkpoint/ckpt.pth')
    net.load_state_dict(checkpoint['net'])
    best_acc = checkpoint['acc']
    start_epoch = checkpoint['epoch']

criterion = nn.CrossEntropyLoss()
# 'We use a weight decay of 0.0001 and momentum of 0.9' (p.7)
optimizer = optim.SGD(net.parameters(), lr=args.lr,
                      momentum=0.9, weight_decay=0.0001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200, eta_min=0.001)

==> Building Model..

==> Model Summary
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 32, 32]             432
       BatchNorm2d-2           [-1, 16, 32, 32]              32
            Conv2d-3           [-1, 16, 32, 32]           2,304
       BatchNorm2d-4           [-1, 16, 32, 32]              32
            Conv2d-5           [-1, 16, 32, 32]           2,304
       BatchNorm2d-6           [-1, 16, 32, 32]              32
        BasicBlock-7           [-1, 16, 32, 32]               0
            Conv2d-8           [-1, 16, 32, 32]           2,304
       BatchNorm2d-9           [-1, 16, 32, 32]              32
           Conv2d-10           [-1, 16, 32, 32]           2,304
      BatchNorm2d-11           [-1, 16, 32, 32]              32
       BasicBlock-12           [-1, 16, 32, 32]               0
           Conv2d-13           [-1, 16, 32, 32]           2,304

In [5]:
# Training
def train(epoch):
    print('\nEpoch: %d' % epoch)
    net.train()
    train_loss = 0
    correct = 0
    total = 0
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        progress_bar(batch_idx, len(trainloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d))'
                     % (train_loss/(batch_idx+1), 100.*correct/total, correct, total))


def test(epoch):
    global best_acc
    net.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            progress_bar(batch_idx, len(testloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d))'
                         % (test_loss/(batch_idx+1), 100.*correct/total, correct, total))

    # Save checkpoint.
    acc = 100.*correct/total
    if acc > best_acc:
        print('Saving..')
        state = {
            'net': net.state_dict(),
            'acc': acc,
            'epoch': epoch,
        }
        if not os.path.isdir('checkpoint'):
            os.mkdir('checkpoint')
        torch.save(state, './checkpoint/ckpt.pth')
        best_acc = acc

In [6]:
for epoch in range(start_epoch, start_epoch+200):
    train(epoch)
    test(epoch)
    scheduler.step()


Epoch: 0
  Step: 289ms | Tot: 15s819ms | Loss: 1.684 | Acc: 36.566% (18283/50000) 391/391 /391 
  Step: 10ms | Tot: 1s9ms | Loss: 1.383 | Acc: 49.420% (4942/10000) 100/100 /100 
Saving..

Epoch: 1
  Step: 42ms | Tot: 15s594ms | Loss: 1.238 | Acc: 54.926% (27463/50000) 391/391 90/391 197/391 209/391 370/391 
  Step: 9ms | Tot: 1s24ms | Loss: 1.220 | Acc: 57.650% (5765/10000) 100/100 
Saving..

Epoch: 2
  Step: 42ms | Tot: 15s458ms | Loss: 0.995 | Acc: 64.562% (32281/50000) 391/391 39 245/39 262/39 263/391 
  Step: 9ms | Tot: 1s26ms | Loss: 1.104 | Acc: 62.660% (6266/10000) 100/100 
Saving..

Epoch: 3
  Step: 40ms | Tot: 15s375ms | Loss: 0.849 | Acc: 70.046% (35023/50000) 391/391 43/391 
  Step: 9ms | Tot: 1s30ms | Loss: 0.983 | Acc: 67.800% (6780/10000) 100/100 
Saving..

Epoch: 4
  Step: 39ms | Tot: 15s392ms | Loss: 0.725 | Acc: 74.534% (37267/50000) 391/391 355/391 
  Step: 9ms | Tot: 1s29ms | Loss: 1.124 | Acc: 65.780% (6578/10000) 100/100 

Epoch: 5
  Step: 39ms | Tot: 15s468ms | L

  Step: 40ms | Tot: 15s468ms | Loss: 0.264 | Acc: 90.818% (45409/50000) 391/391 7/39 244/391 
  Step: 10ms | Tot: 1s18ms | Loss: 0.448 | Acc: 85.760% (8576/10000) 100/100 

Epoch: 40
  Step: 40ms | Tot: 15s432ms | Loss: 0.259 | Acc: 90.950% (45475/50000) 391/391  335/39 375/391 
  Step: 10ms | Tot: 1s94ms | Loss: 0.422 | Acc: 86.370% (8637/10000) 100/100 100 
Saving..

Epoch: 41
  Step: 40ms | Tot: 15s415ms | Loss: 0.259 | Acc: 90.954% (45477/50000) 391/391 287/39 353/391 
  Step: 10ms | Tot: 1s35ms | Loss: 0.524 | Acc: 83.920% (8392/10000) 100/100 

Epoch: 42
  Step: 40ms | Tot: 15s559ms | Loss: 0.259 | Acc: 90.814% (45407/50000) 391/391 3/391 246/391 
  Step: 9ms | Tot: 1s23ms | Loss: 0.415 | Acc: 86.900% (8690/10000) 100/100 
Saving..

Epoch: 43
  Step: 40ms | Tot: 15s472ms | Loss: 0.256 | Acc: 90.928% (45464/50000) 391/391 
  Step: 9ms | Tot: 1s38ms | Loss: 0.421 | Acc: 86.840% (8684/10000) 100/100 

Epoch: 44
  Step: 39ms | Tot: 15s626ms | Loss: 0.252 | Acc: 91.176% (45588/50000) 

  Step: 40ms | Tot: 15s543ms | Loss: 0.181 | Acc: 93.634% (46817/50000) 391/391  23/391  93/39 239/391 
  Step: 9ms | Tot: 1s27ms | Loss: 0.490 | Acc: 85.810% (8581/10000) 100/100 100 

Epoch: 82
  Step: 39ms | Tot: 15s557ms | Loss: 0.179 | Acc: 93.788% (46894/50000) 391/391 39 182/39 301/391 311/39 323/391 344/391 
  Step: 9ms | Tot: 1s20ms | Loss: 0.373 | Acc: 88.800% (8880/10000) 100/100 
Saving..

Epoch: 83
  Step: 39ms | Tot: 15s566ms | Loss: 0.176 | Acc: 93.792% (46896/50000) 391/391 89/39 168/39 213/39 323/391 
  Step: 9ms | Tot: 1s28ms | Loss: 0.385 | Acc: 88.310% (8831/10000) 100/100 

Epoch: 84
  Step: 40ms | Tot: 15s544ms | Loss: 0.174 | Acc: 93.826% (46913/50000) 391/391 0/391 182/391 246/391 349/391 352/391 372/391 
  Step: 9ms | Tot: 1s27ms | Loss: 0.375 | Acc: 88.400% (8840/10000) 100/100 

Epoch: 85
  Step: 41ms | Tot: 15s622ms | Loss: 0.174 | Acc: 93.850% (46925/50000) 391/391 40/39 107/39 126/391 127/391 271/391 293/391 
  Step: 9ms | Tot: 1s21ms | Loss: 0.381 | Acc: 

  Step: 42ms | Tot: 15s550ms | Loss: 0.098 | Acc: 96.602% (48301/50000) 391/391 4/391 214/39 286/391 
  Step: 10ms | Tot: 1s59ms | Loss: 0.398 | Acc: 89.480% (8948/10000) 100/100 00 

Epoch: 122
  Step: 40ms | Tot: 15s575ms | Loss: 0.096 | Acc: 96.582% (48291/50000) 391/391 9 37/391 61/39 176/391 195/391 337/391 
  Step: 10ms | Tot: 1s31ms | Loss: 0.398 | Acc: 89.390% (8939/10000) 100/100 

Epoch: 123
  Step: 39ms | Tot: 15s406ms | Loss: 0.096 | Acc: 96.590% (48295/50000) 391/391  86/39 93/391 
  Step: 9ms | Tot: 1s21ms | Loss: 0.395 | Acc: 89.870% (8987/10000) 100/100 

Epoch: 124
  Step: 40ms | Tot: 15s412ms | Loss: 0.097 | Acc: 96.578% (48289/50000) 391/391 /391 
  Step: 10ms | Tot: 1s104ms | Loss: 0.410 | Acc: 89.220% (8922/10000) 100/100 

Epoch: 125
  Step: 39ms | Tot: 15s433ms | Loss: 0.092 | Acc: 96.722% (48361/50000) 391/391 391 142/391 
  Step: 10ms | Tot: 1s52ms | Loss: 0.392 | Acc: 89.680% (8968/10000) 100/100 

Epoch: 126
  Step: 40ms | Tot: 15s484ms | Loss: 0.087 | Acc: 9

  Step: 10ms | Tot: 1s30ms | Loss: 0.386 | Acc: 91.460% (9146/10000) 100/100 00 

Epoch: 163
  Step: 40ms | Tot: 15s622ms | Loss: 0.019 | Acc: 99.444% (49722/50000) 391/391 1 
  Step: 9ms | Tot: 1s23ms | Loss: 0.386 | Acc: 91.410% (9141/10000) 100/100 

Epoch: 164
  Step: 41ms | Tot: 15s516ms | Loss: 0.019 | Acc: 99.458% (49729/50000) 391/391  66/39 336/39 343/391 
  Step: 10ms | Tot: 1s44ms | Loss: 0.388 | Acc: 91.470% (9147/10000) 100/100 100 

Epoch: 165
  Step: 39ms | Tot: 15s529ms | Loss: 0.018 | Acc: 99.492% (49746/50000) 391/391 9 128/391 
  Step: 10ms | Tot: 1s22ms | Loss: 0.382 | Acc: 91.340% (9134/10000) 100/100 

Epoch: 166
  Step: 41ms | Tot: 15s628ms | Loss: 0.017 | Acc: 99.516% (49758/50000) 391/391 3/391 192/39 284/391 349/391 
  Step: 9ms | Tot: 1s50ms | Loss: 0.396 | Acc: 91.330% (9133/10000) 100/100 

Epoch: 167
  Step: 39ms | Tot: 15s941ms | Loss: 0.017 | Acc: 99.540% (49770/50000) 391/391 0/39 263/391 
  Step: 10ms | Tot: 1s32ms | Loss: 0.386 | Acc: 91.390% (9139/10

In [7]:
print('Accuracy:', round(best_acc, 2))
print('Error:', round(100-best_acc, 2))

Accuracy: 92.0
Error: 8.0
